## 01_data_exploration.ipynb

In [56]:
import os
import polars as pl

batch = os.getcwd()
path = os.path.join(batch, '..', 'data', 'kidney_disease.csv')
path = os.path.normpath(path)
 
df = pl.read_csv(
    path,
    null_values=["?", "", "NA", "N/A", "nan", "NaN"],
    infer_schema_length=10000,
    ignore_errors=True
)

print("Loaded Successfully!")
print("Shape        :", df.shape)
print("Null counts  :")
print(df.null_count())
print(df.head())

Loaded Successfully!
Shape        : (400, 26)
Null counts  :
shape: (1, 26)
┌─────┬─────┬─────┬─────┬───┬───────┬─────┬─────┬────────────────┐
│ id  ┆ age ┆ bp  ┆ sg  ┆ … ┆ appet ┆ pe  ┆ ane ┆ classification │
│ --- ┆ --- ┆ --- ┆ --- ┆   ┆ ---   ┆ --- ┆ --- ┆ ---            │
│ u32 ┆ u32 ┆ u32 ┆ u32 ┆   ┆ u32   ┆ u32 ┆ u32 ┆ u32            │
╞═════╪═════╪═════╪═════╪═══╪═══════╪═════╪═════╪════════════════╡
│ 0   ┆ 9   ┆ 12  ┆ 47  ┆ … ┆ 1     ┆ 1   ┆ 1   ┆ 0              │
└─────┴─────┴─────┴─────┴───┴───────┴─────┴─────┴────────────────┘
shape: (5, 26)
┌─────┬──────┬──────┬───────┬───┬───────┬─────┬─────┬────────────────┐
│ id  ┆ age  ┆ bp   ┆ sg    ┆ … ┆ appet ┆ pe  ┆ ane ┆ classification │
│ --- ┆ ---  ┆ ---  ┆ ---   ┆   ┆ ---   ┆ --- ┆ --- ┆ ---            │
│ i64 ┆ f64  ┆ f64  ┆ f64   ┆   ┆ str   ┆ str ┆ str ┆ str            │
╞═════╪══════╪══════╪═══════╪═══╪═══════╪═════╪═════╪════════════════╡
│ 0   ┆ 48.0 ┆ 80.0 ┆ 1.02  ┆ … ┆ good  ┆ no  ┆ no  ┆ ckd            │
│ 1   ┆ 7.0  ┆

## 02_data_cleaning.ipynb

In [57]:
df.null_count()

id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,9,12,47,46,49,152,65,4,4,44,19,17,87,88,52,70,105,130,2,2,2,1,1,1,0


In [58]:
def fill_numeric_with_median(df: pl.DataFrame) -> pl.DataFrame:
    import polars.selectors as cs
    return df.with_columns(
        cs.numeric().fill_null(cs.numeric().median())
    )

df = fill_numeric_with_median(df)

In [59]:
def fill_string_with_mode(df: pl.DataFrame) -> pl.DataFrame:
    import polars.selectors as cs
    return df.with_columns(
        cs.string().fill_null(cs.string().mode())
    )

df = fill_string_with_mode(df)

In [60]:
import polars as pl

num_cols = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc", 
            "sod", "pot", "hemo", "pcv", "wc", "rc"]

df = df.with_columns([
    pl.col(col)
      .cast(pl.Utf8)                  # ensure string first
      .str.strip_chars()              # remove whitespace/tabs
      .str.replace(r"[^\d.]", "")    # remove non-numeric chars like "?"
      .replace("", None)             # empty string → null
      .cast(pl.Float64)              # now safely cast to numeric
    for col in num_cols
])

# Now median fill works ✅
df = df.with_columns([
    pl.col(col).fill_null(pl.col(col).median())
    for col in num_cols
])

In [61]:
df.null_count()

id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 03_eda.ipynb

In [62]:
import plotly.express as px

fig = px.box(
    data_frame=df,
    x="age",
    y="bp",
    color="classification",
    boxmode="group",
    notched=True,
    points="all",
    hover_name="classification",
    hover_data={
        "age": True,
        "al": True,
        "bp": True,
    },
    # facet_col="classification",  
    orientation="v",
    title="Kidney Disease - Age vs Blood Pressure",
    labels={
        "age": "Age",
        "bp": "Blood Pressure",
        "classification": "CKD Status",
    },
    category_orders={
        "classification": ["ckd", "notckd"]  
    },
    color_discrete_map={
        "ckd":    "#EF553B",   
        "notckd": "#636EFA",
    },
    template="plotly_white",
    width=950,
    height=600,
)

fig.update_traces(
    jitter=0.3,
    pointpos=-1.5,
    opacity=0.8,
)

fig.show()

In [63]:
import plotly.express as px

# Box plot — shows outlier dots automatically
fig = px.box(df, y="bp", points="outliers", title="BP Outliers")
fig.show()

# For all numeric columns
import polars.selectors as cs
num_cols = df.select(cs.numeric()).columns

fig = px.box(df.to_pandas()[num_cols], title="All Numeric Outliers")
fig.show()

In [64]:
import polars as pl
import polars.selectors as cs

def remove_outliers_zscore(df: pl.DataFrame, col: str, threshold=3) -> pl.DataFrame:
    mean = df[col].mean()
    std  = df[col].std()
    return df.filter(
        ((pl.col(col) - mean) / std).abs() < threshold
    )

num_cols = df.select(cs.numeric()).columns

df_clean = df.clone()
for col in num_cols:
    df_clean = remove_outliers_zscore(df_clean, col=col)

In [65]:
num_cols = df_clean.select(cs.numeric()).columns
fig = px.box(df_clean.to_pandas()[num_cols], title="All Numeric Outliers")
fig.show()

## 04_feature_engineering.ipynb

In [66]:
cat_col = df_clean.select(cs.string()).columns
cat_col

['rbc',
 'pc',
 'pcc',
 'ba',
 'htn',
 'dm',
 'cad',
 'appet',
 'pe',
 'ane',
 'classification']

In [67]:
import polars.selectors as cs

cat_cols = df_clean.select(cs.string()).columns

for col in cat_cols:
    print(f"\n── {col} ──")
    print(df[col].value_counts(sort=True))


── rbc ──
shape: (2, 2)
┌──────────┬───────┐
│ rbc      ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ normal   ┆ 353   │
│ abnormal ┆ 47    │
└──────────┴───────┘

── pc ──
shape: (2, 2)
┌──────────┬───────┐
│ pc       ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ normal   ┆ 324   │
│ abnormal ┆ 76    │
└──────────┴───────┘

── pcc ──
shape: (2, 2)
┌────────────┬───────┐
│ pcc        ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ notpresent ┆ 358   │
│ present    ┆ 42    │
└────────────┴───────┘

── ba ──
shape: (2, 2)
┌────────────┬───────┐
│ ba         ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ notpresent ┆ 378   │
│ present    ┆ 22    │
└────────────┴───────┘

── htn ──
shape: (2, 2)
┌─────┬───────┐
│ htn ┆ count │
│ --- ┆ ---   │
│ str ┆ u32   │
╞═════╪═══════╡
│ no  ┆ 253   │
│ yes ┆ 147   │
└─────┴───────┘

── dm ──
shape: (5, 2)
┌──────┬───────┐
│ dm

In [68]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV


In [69]:
X, y = df_clean.drop('classification'), df_clean['classification']

In [70]:
num_col = X.select(cs.numeric()).columns
cat_col = X.select(cs.string()).columns


len(cat_col), len(num_col)

(10, 15)

In [71]:
X.head(), y.head()

(shape: (5, 25)
 ┌─────┬──────┬───────┬───────┬───┬─────┬───────┬─────┬─────┐
 │ id  ┆ age  ┆ bp    ┆ sg    ┆ … ┆ cad ┆ appet ┆ pe  ┆ ane │
 │ --- ┆ ---  ┆ ---   ┆ ---   ┆   ┆ --- ┆ ---   ┆ --- ┆ --- │
 │ f64 ┆ f64  ┆ f64   ┆ f64   ┆   ┆ str ┆ str   ┆ str ┆ str │
 ╞═════╪══════╪═══════╪═══════╪═══╪═════╪═══════╪═════╪═════╡
 │ 0.0 ┆ 48.0 ┆ 80.0  ┆ 1.02  ┆ … ┆ no  ┆ good  ┆ no  ┆ no  │
 │ 1.0 ┆ 7.0  ┆ 50.0  ┆ 1.02  ┆ … ┆ no  ┆ good  ┆ no  ┆ no  │
 │ 4.0 ┆ 51.0 ┆ 80.0  ┆ 1.01  ┆ … ┆ no  ┆ good  ┆ no  ┆ no  │
 │ 5.0 ┆ 60.0 ┆ 90.0  ┆ 1.015 ┆ … ┆ no  ┆ good  ┆ yes ┆ no  │
 │ 8.0 ┆ 52.0 ┆ 100.0 ┆ 1.015 ┆ … ┆ no  ┆ good  ┆ no  ┆ yes │
 └─────┴──────┴───────┴───────┴───┴─────┴───────┴─────┴─────┘,
 shape: (10,)
 Series: 'classification' [str]
 [
 	"ckd"
 	"ckd"
 	"ckd"
 	"ckd"
 	"ckd"
 	"ckd"
 	"ckd"
 	"ckd"
 	"ckd"
 	"ckd"
 ])

In [72]:
encode = OrdinalEncoder(categories=[
    ['normal', 'abnormal'],
    ['normal', 'abnormal'],
    ['present', 'notpresent'],
    ['present', 'notpresent'],
    ['no', 'yes'],
    ['no', 'yes'],
    ['no', 'yes'],
    ['good', 'poor'],
    ['no', 'yes'],
    ['no', 'yes'],  
],handle_unknown="use_encoded_value", unknown_value=-1)

scaler = StandardScaler()

In [73]:
transform = ColumnTransformer(
    transformers=[
        ('ordinal', encode, cat_col),
        ('scaler',scaler , num_col),
    ]
)

In [74]:
# Clean all string columns in Polars
X = X.with_columns(
    pl.col(cat_col).str.strip_chars()
)

In [75]:
import pandas as pd
x_pd = X.to_pandas()
# result = transform.fit_transform(x_pd
#                                  )
# print(result.shape)   # should be (rows, expected_cols)
# print(result[:5])     # first 5 rows


In [76]:
from sklearn.preprocessing import LabelEncoder

# Strip directly on the Series
y = y.str.strip_chars()

# Encode
y_encoded = LabelEncoder().fit_transform(y)

In [77]:
X_train, X_test, y_train, y_test = train_test_split(x_pd, y_encoded, test_size=0.2, random_state=42)


In [78]:
len(X_train.columns)

25

In [79]:
X_train_scaled = transform.fit_transform(X_train)
X_test_scaled = transform.transform(X_test)

In [80]:
X_train_scaled.shape

(272, 25)

## 05_model_building.ipynb

In [81]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
import numpy as np


In [82]:
dt_params = {
    "criterion":        ["gini", "entropy", "log_loss"],
    "splitter":         ["best", "random"],
    "max_depth":        [None, 3, 5, 10, 20],
    "min_samples_split":[2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features":     [None, "sqrt", "log2"],
    "max_leaf_nodes":   [None, 10, 20, 50],
    "class_weight":     [None, "balanced"],
}

mnb_params = {
    "alpha":     [0.0, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
    "fit_prior": [True, False],
}

gnb_params = {
    "var_smoothing": np.logspace(-11, -5, 30),   # 30 log-spaced values
}

In [83]:
dt_mode = RandomizedSearchCV(
    DecisionTreeClassifier(),
    param_distributions=dt_params,
    n_jobs=-1,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    random_state=42

)

dt_mode.fit(X_train_scaled, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeClassifier()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'class_weight': [None, 'balanced'], 'criterion': ['gini', 'entropy', ...], 'max_depth': [None, 3, ...], 'max_features': [None, 'sqrt', ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be 

In [84]:
dt_mode.score(X_test_scaled, y_test)

1.0

In [85]:
# mnb_mode = RandomizedSearchCV(
#     MultinomialNB(),
#     param_distributions=mnb_params,
#     n_jobs=-1,
#     n_iter=10,
#     cv=5,
#     scoring='accuracy',
#     random_state=42

# )

# mnb_mode.fit(X_train_scaled, y_train)

In [86]:
gnb_mode = RandomizedSearchCV(
    GaussianNB(),
    param_distributions=gnb_params,
    n_jobs=-1,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    random_state=42

)

gnb_mode.fit(X_train_scaled, y_train)

c:\Users\sumit\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning:

The total space of parameters 30 is smaller than n_iter=50. Running 30 iterations. For exhaustive searches, use GridSearchCV.



,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",GaussianNB()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.",{'var_smoothing': array([1.0000...00000000e-05])}
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose ver

In [87]:
gnb_mode.score(X_test_scaled, y_test)

0.9852941176470589

In [88]:
print(dt_mode.best_params_)
print(gnb_mode.best_params_)

{'splitter': 'best', 'min_samples_split': 10, 'min_samples_leaf': 10, 'max_leaf_nodes': 20, 'max_features': None, 'max_depth': 5, 'criterion': 'gini', 'class_weight': None}
{'var_smoothing': np.float64(1e-11)}


In [89]:
model_dt = dt_mode.best_estimator_
y_pred = model_dt.predict(X_test_scaled)

In [90]:
testing = pl.DataFrame({'true':y_test[:10], 'pred': y_pred[:10]})
print(testing)

shape: (10, 2)
┌──────┬──────┐
│ true ┆ pred │
│ ---  ┆ ---  │
│ i64  ┆ i64  │
╞══════╪══════╡
│ 1    ┆ 1    │
│ 0    ┆ 0    │
│ 0    ┆ 0    │
│ 0    ┆ 0    │
│ 0    ┆ 0    │
│ 1    ┆ 1    │
│ 0    ┆ 0    │
│ 1    ┆ 1    │
│ 1    ┆ 1    │
│ 0    ┆ 0    │
└──────┴──────┘


In [91]:
model_gnb = gnb_mode.best_estimator_
y_pred_1 = model_gnb.predict(X_test_scaled)

In [92]:
testing_1 = pl.DataFrame({'true':y_test[:10], 'pred': y_pred[:10]})
print(testing_1)

shape: (10, 2)
┌──────┬──────┐
│ true ┆ pred │
│ ---  ┆ ---  │
│ i64  ┆ i64  │
╞══════╪══════╡
│ 1    ┆ 1    │
│ 0    ┆ 0    │
│ 0    ┆ 0    │
│ 0    ┆ 0    │
│ 0    ┆ 0    │
│ 1    ┆ 1    │
│ 0    ┆ 0    │
│ 1    ┆ 1    │
│ 1    ┆ 1    │
│ 0    ┆ 0    │
└──────┴──────┘


## model comparison for matrix methods

In [93]:
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, recall_score, precision_score
import plotly.graph_objects as go

dt_confusion_matrix = confusion_matrix(y_test, y_pred)
gnb_confusion_matrix = confusion_matrix(y_test, y_pred_1)
dt_accuracy = accuracy_score(y_test, y_pred)
gnb_accuracy = accuracy_score(y_test, y_pred_1)
dt_f1Score = f1_score(y_test, y_pred)
gnb_f1Score= f1_score(y_test, y_pred_1)
dt_recall = recall_score(y_test, y_pred)
gnb_recall = recall_score(y_test, y_pred_1)
dt_precison = precision_score(y_test, y_pred)
gnb_precison = precision_score(y_test, y_pred_1)


metrics_df = pl.DataFrame({
    "model":     ["Decision Tree", "Gaussian NB"],
    "accuracy":  [dt_accuracy,     gnb_accuracy],
    "f1_score":  [dt_f1Score,      gnb_f1Score],
    "recall":    [dt_recall,       gnb_recall],
    "precision": [dt_precison,     gnb_precison],
})




In [98]:
import os
import plotly.io as pio

# --- Grouped Bar Chart ---
metrics = ["accuracy", "f1_score", "recall", "precision"]
colors  = ["#4C9BE8", "#E8834C"]

fig = go.Figure()

for i, row in enumerate(metrics_df.iter_rows(named=True)):
    fig.add_trace(go.Bar(
        name=row["model"],
        x=metrics,
        y=[row["accuracy"], row["f1_score"], row["recall"], row["precision"]],
        marker_color=colors[i],
        text=[f"{v:.2f}" for v in [row["accuracy"], row["f1_score"], row["recall"], row["precision"]]],
        textposition="outside",
    ))

fig.update_layout(
    title="Model Metrics Comparison",
    barmode="group",
    xaxis_title="Metric",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1.15]),
    legend_title="Model",
    height=500,
    width=750,
    template="plotly_white",
)

# --- Save Image ---
batch = os.getcwd()
path  = os.path.join(batch, '..', 'outputs', 'figures', 'comparison_table.png')   

os.makedirs(os.path.dirname(path), exist_ok=True)    

fig.write_image(path)    
fig.show()

In [100]:
import plotly.figure_factory as ff
import plotly.graph_objects as go

import numpy as np

# --- After predictions ---
 
labels =  ["ckd", "notckd"]

cm = confusion_matrix(y_test, y_pred)

# --- Plotly Heatmap ---
fig1 = ff.create_annotated_heatmap(
    z=cm,
    x=list(labels),           # predicted (columns)
    y=list(labels),            # actual (rows)
    annotation_text=cm,        # show numbers in cells
    colorscale="Blues",
    showscale=True,
)

# --- Labels ---
fig1.update_layout(
    title="Confusion Matrix",
    xaxis=dict(title="Predicted Label", side="bottom"),
    yaxis=dict(title="Actual Label", autorange="reversed"),  # top-to-bottom
    width=500,
    height=450,
)

fig1.show()

In [101]:
labels =  ["ckd", "notckd"]

cm = confusion_matrix(y_test, y_pred_1)

# --- Plotly Heatmap ---
fig = ff.create_annotated_heatmap(
    z=cm,
    x=list(labels),           # predicted (columns)
    y=list(labels),            # actual (rows)
    annotation_text=cm,        # show numbers in cells
    colorscale="Blues",
    showscale=True,
)

# --- Labels ---
fig.update_layout(
    title="Confusion Matrix",
    xaxis=dict(title="Predicted Label", side="bottom"),
    yaxis=dict(title="Actual Label", autorange="reversed"),  # top-to-bottom
    width=500,
    height=450,
)

fig.show()

## save martix 

In [102]:
batch = os.getcwd()
path  = os.path.join(batch, '..', 'outputs', 'figures', 'confusion_gnb.png')
path1  = os.path.join(batch, '..', 'outputs', 'figures', 'confusion_dt.png')

os.makedirs(os.path.dirname(path), exist_ok=True)    

fig.write_image(path)  
fig1.write_image(path1)

## save column transformer

In [106]:
import joblib as jl
batch = os.getcwd()
path  = os.path.join(batch,'..' ,'outputs','models', 'encoding.pkl')
os.makedirs(os.path.dirname(path), exist_ok=True)
jl.dump(transform, path)


['c:\\Users\\sumit\\OneDrive\\Desktop\\project work\\kidney-failure-detection\\notebook\\..\\outputs\\models\\encoding.pkl']

## model make for good parms fully trained

In [107]:
model_dt = DecisionTreeClassifier(splitter='best', min_samples_split=10, min_samples_leaf = 10, max_leaf_nodes = 20, max_features = None, max_depth= 5, criterion='gini', class_weight = None)


model_gnb = GaussianNB(var_smoothing = np.float64(1e-11))

In [108]:
model_dt.fit(X_train_scaled, y_train)
model_gnb.fit(X_train_scaled, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",np.float64(1e-11)


In [110]:
model_gnb.score(X_test_scaled, y_test),model_dt.score(X_test_scaled, y_test)


(0.9852941176470589, 1.0)

In [113]:
batch = os.getcwd()
path_dt  = os.path.join(batch,'..' ,'outputs','models', 'model_dt.pkl')
path_gnb  = os.path.join(batch,'..' ,'outputs','models', 'model_gnb.pkl')
os.makedirs(os.path.dirname(path), exist_ok=True)

jl.dump(model_dt,path_dt)
jl.dump(model_gnb, path_gnb)

['c:\\Users\\sumit\\OneDrive\\Desktop\\project work\\kidney-failure-detection\\notebook\\..\\outputs\\models\\model_gnb.pkl']